# ML Book Reader — PDF Extraction (Google Colab)

Extracts the 5 chapter PDFs of Prof. Reza Rawassizadeh's ML textbook into markdown + images using [marker-pdf](https://github.com/datalab-to/marker).

**Why Colab?** Marker needs ~5 GB of GPU/RAM for its layout + text-recognition + OCR-error models. An 8 GB M1 Air swaps itself to death. Colab's free tier gives us a T4 GPU + 12 GB RAM and runs the whole book in ~15–20 min.

## How to run

1. Open this notebook in Colab (File → Upload notebook, or open from GitHub once pushed).
2. **Runtime → Change runtime type → T4 GPU** (free tier). Confirm.
3. Run cells top-to-bottom.
4. When prompted, upload the 5 chapter PDFs from your local `~/MLBook/` folder.
5. Last cell downloads `mlbook-extracted.zip` to your Mac.
6. Unzip into `ml-book-reader/tmp/extracted/` so the layout matches what `scripts/split-sections.ts` expects:
   ```
   tmp/extracted/
     ch3/<stem>/<stem>.md + images
     ch8/<stem>/<stem>.md + images
     ...
   ```
7. Back on the Mac:
   ```
   cd ~/ml-book-reader
   npx tsx scripts/split-sections.ts
   bash scripts/upload-r2.sh
   npx tsx scripts/index-book.ts
   ```

## 1. Verify GPU is attached

In [ ]:
!nvidia-smi

Expected output: a table showing `Tesla T4` with ~15 GB memory. If you see `No GPU`, go to Runtime → Change runtime type → T4 GPU.

## 2. Install marker-pdf

We install the latest `marker-pdf` (no pin) because `surya-ocr` downloads its model checkpoints from `models.datalab.to` at runtime and grabs the newest schema. Older marker versions expected an older checkpoint layout (missing `encoder` key) and crash with `KeyError: 'encoder'`. First install takes ~2 min.

**If you already ran a broken pin earlier in this session:** Runtime → Disconnect and delete runtime, then start fresh. A dirty pip state is the most common cause of re-appearing errors.

In [ ]:
!pip install -q --upgrade pip
!pip install -q --upgrade marker-pdf
!pip show marker-pdf | grep -E '^(Name|Version)'
!pip show surya-ocr | grep -E '^(Name|Version)'

## 3. Upload the 5 chapter PDFs

Select **all 5 PDFs at once** from your `~/MLBook/` folder in the file picker. Expected names:

- `Chapter 3_Probability and Statistics.pdf`
- `Chapter 8 -Regressions.pdf`
- `Chapter 9- Classification Algorithms.pdf`
- `Chapter 10-Artificial Neural Network.pdf`
- `Chapter 11-Self-supervised Deep Learning.pdf`

In [ ]:
import os, shutil
from google.colab import files

os.makedirs('/content/pdfs', exist_ok=True)
os.makedirs('/content/extracted', exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f'/content/pdfs/{name}')

print('\nUploaded PDFs:')
!ls -lh /content/pdfs/

## 4. Run marker on every PDF

Maps each PDF to a chapter id so the output layout matches the repo expectations. Extraction runs sequentially (marker already uses the GPU internally). Expect ~3–5 min per chapter.

In [ ]:
# Chapter id -> PDF filename mapping. Keep in sync with scripts/extract-pdfs.sh.
CHAPTERS = {
    'ch3':  'Chapter 3_Probability and Statistics.pdf',
    'ch8':  'Chapter 8 -Regressions.pdf',
    'ch9':  'Chapter 9- Classification Algorithms.pdf',
    'ch10': 'Chapter 10-Artificial Neural Network.pdf',
    'ch11': 'Chapter 11-Self-supervised Deep Learning.pdf',
}

# Sanity-check all 5 PDFs are present before we start a long run.
missing = [f for f in CHAPTERS.values() if not os.path.exists(f'/content/pdfs/{f}')]
if missing:
    raise SystemExit(f'Missing PDFs in /content/pdfs: {missing}. Re-run the upload cell.')

print('All 5 PDFs present. Starting extraction.\n')

In [ ]:
import subprocess, time

for ch, fname in CHAPTERS.items():
    out_dir = f'/content/extracted/{ch}'
    os.makedirs(out_dir, exist_ok=True)
    pdf_path = f'/content/pdfs/{fname}'
    print(f'=== [{ch}] {fname} ===')
    t0 = time.time()
    # `marker_single` uses GPU automatically when CUDA is available.
    result = subprocess.run(
        ['marker_single', pdf_path, '--output_dir', out_dir, '--output_format', 'markdown'],
        capture_output=True, text=True,
    )
    dt = time.time() - t0
    if result.returncode != 0:
        print(f'  !! FAILED (exit {result.returncode}) after {dt:.1f}s')
        print(result.stderr[-2000:])
        raise SystemExit(f'{ch} failed; fix and re-run.')
    print(f'  done in {dt:.1f}s')

print('\nAll chapters extracted.')

## 5. Inspect the output

In [ ]:
!find /content/extracted -maxdepth 4 -name '*.md' -exec ls -lh {} \;
print()
!du -sh /content/extracted/*

## 6. Zip + download

Produces `mlbook-extracted.zip`. After download, unzip into `ml-book-reader/tmp/extracted/` so the layout matches `scripts/split-sections.ts`.

In [ ]:
!cd /content && zip -qr mlbook-extracted.zip extracted
!ls -lh /content/mlbook-extracted.zip

from google.colab import files
files.download('/content/mlbook-extracted.zip')

## 7. On your Mac

```bash
cd ~/ml-book-reader
rm -rf tmp/extracted
mkdir -p tmp
unzip ~/Downloads/mlbook-extracted.zip -d tmp/
# Verify layout: tmp/extracted/ch3/<stem>/<stem>.md should exist
find tmp/extracted -maxdepth 4 -name '*.md'

# Then continue the pipeline:
npx tsx scripts/split-sections.ts
bash scripts/upload-r2.sh
npx tsx scripts/index-book.ts
```